In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re
import time
from datetime import datetime, timedelta
import os

# Initialize storage for article data and Parquet file path
article_data = []
parquet_file = "../../data/00-newspaper_data/crawler/tabascohoy/articles.parquet"

# Base URL format for categories
base_url = "https://www.tabascohoy.com/{category}/page/{page}/"

# Define categories to scrape
categories = {
    "Tab": {"cat": "tabasco", "max_pages": 3121},
    "Seguridad": {"cat": "seguridad", "max_pages": 1149},
    "Mexico": {"cat": "mexico", "max_pages": 2770},
    "Mundo": {"cat": "mundo", "max_pages": 871},
    "Dinero": {"cat": "dinero", "max_pages": 152},
    "Opinion": {"cat": "tag/columnista", "max_pages": 580},
    "Deportes": {"cat": "100-deportes", "max_pages": 1255},
    "Like": {"cat": "like", "max_pages": 1129},
    "Aplausos": {"cat": "aplausos", "max_pages": 20},
    "Tecno": {"cat": "tecnologia", "max_pages": 210},
    "Unis": {"cat": "unis-y-mas", "max_pages": 138}
}


MAX_RETRIES = 3

def is_relevant_url(url):
    """
    Check if the URL matches the pattern YYYY/MM/DD/title.
    """
    pattern = r'https?://quintanaroohoy\.com/(?!author|category|tag)[a-zA-Z0-9-]+/[a-zA-Z0-9-]+/?$'
    return re.search(pattern, url)


def extract_article_data(url):
    """
    Extract and return the title, main text, date, and source from an article page.
    """
    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')

        # Extract the title
        try: 
            title_tag = soup.find('h1', class_='zox-post-title left entry-title')
            title = title_tag.get_text(strip=True) if title_tag else 'No title found'
        except:
            title = None

        # Extract the main text
        try:
            dd_content = soup.find_all('div', class_='zox-post-main')
            main_text = ' '.join(p.get_text(strip=True) for div in dd_content for p in div.find_all('p'))
        except:
            main_text = None

        try:
            subtitle_tag = soup.find('span', class_='zox-post-excerpt')
            subtitle = ' '.join(p.get_text(strip=True) for div in subtitle_tag for p in div.find_all('p'))
        except: 
            subtitle = None

        try:
            date_tag = soup.find('time')
            date = date_tag['datetime'] if date_tag and 'datetime' in date_tag.attrs else 'No date found'
        except: 
            date = None
        # Extract the source

        try:
            category_tag = soup.find('span', class_='zox-post-cat')
            category = category_tag.get_text(strip=True) if title_tag else 'No category found'
        except: 
            category = None

        return {
            'url': url,
            'title': title,
            'sub_title':subtitle,
            'main_text': main_text.strip(),
            'date': date,
            'topic': category
        }
    
    except requests.exceptions.RequestException as e:
        print(f"Error fetching data from {url}: {e}")
        return None


def crawl_category(category, maxpag):
    """
    Crawl pages 1 to max (manually inputed for each category) in a given category, extracting article links and data.
    """
    for page in range(1, maxpag + 1):
        category_url = base_url.format(category=category, page=page)
        print(f"Accessing category page: {category_url}")

        retries = 0
        while retries < MAX_RETRIES:
            try:
                response = requests.get(category_url, timeout=15)
                if response.status_code != 200:
                    print(f"Page {page} does not exist in category {category}. Moving to next page.")
                    break  # Stop retrying and move to the next page
                
                soup = BeautifulSoup(response.text, 'html.parser')

                # Find and process all article links on the page
                unique_links = set()
                for link in soup.find_all("a", href=True):
                    href = urljoin(category_url, link['href'])
                    
                    if is_relevant_url(href):
                        unique_links.add(href)  # Store unique article URLs

                # Extract article data immediately
                category_data = []
                for link in unique_links:
                    article = extract_article_data(link)
                    if article:
                        category_data.append(article)
                        print(f"Extracted article from {link}")

                # Save to Parquet
                if category_data:
                    save_to_parquet(category_data, parquet_file)

                time.sleep(2)  # Avoid overloading the server
                break  # If the request was successful, break the retry loop

            except requests.exceptions.RequestException as e:
                retries += 1
                print(f"Connection error on {category_url}. Retrying ({retries}/{MAX_RETRIES})...")
                time.sleep(5)  # Wait before retrying
        
        if retries == MAX_RETRIES:
            print(f"Skipping page {page} in category {category} after {MAX_RETRIES} failed attempts.")
            continue  # Move to the next page even if this one failed


# Function to save data to Parquet
def save_to_parquet(data, parquet_file):
    df = pd.DataFrame(data)
    if not df.empty:
        if os.path.exists(parquet_file):
            initial = pd.read_parquet(parquet_file)
            df = pd.concat([initial, df]).reset_index(drop=True)
            df.to_parquet(parquet_file, engine="pyarrow", compression="gzip")
        else:
            df.to_parquet(parquet_file, engine="pyarrow", compression="gzip")


# Crawl all categories and extract data
for category, info in categories.items():
    print(f"Starting crawl for category: {category}")
    crawl_category(info['cat'], info['max_pages'])

print("Crawling completed. Data saved in Parquet format.")

Starting crawl for category: Tab
Accessing category page: https://www.tabascohoy.com/tabasco/page/1/
Page 1 does not exist in category tabasco. Moving to next page.
Accessing category page: https://www.tabascohoy.com/tabasco/page/2/
Page 2 does not exist in category tabasco. Moving to next page.
Accessing category page: https://www.tabascohoy.com/tabasco/page/3/
Page 3 does not exist in category tabasco. Moving to next page.
Accessing category page: https://www.tabascohoy.com/tabasco/page/4/
Page 4 does not exist in category tabasco. Moving to next page.
Accessing category page: https://www.tabascohoy.com/tabasco/page/5/
Page 5 does not exist in category tabasco. Moving to next page.
Accessing category page: https://www.tabascohoy.com/tabasco/page/6/
Page 6 does not exist in category tabasco. Moving to next page.
Accessing category page: https://www.tabascohoy.com/tabasco/page/7/
Page 7 does not exist in category tabasco. Moving to next page.
Accessing category page: https://www.tabasc

KeyboardInterrupt: 

In [38]:
import undetected_chromedriver as uc

options = uc.ChromeOptions()
options.headless = False  # Running in visible mode for debugging
driver = uc.Chrome(options=options)

driver.get('https://www.tabascohoy.com/ignoro-el-por-que-balearon-a-mi-marido/')
print(driver.page_source)
driver.quit()

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1000)>

In [35]:
driver.get('https://www.tabascohoy.com/ignoro-el-por-que-balearon-a-mi-marido/')

In [32]:
from scrapy import Selector

url = 'https://www.tabascohoy.com/ignoro-el-por-que-balearon-a-mi-marido/'
response = requests.get(url, timeout=15).content

sel = Selector(text = response)
title = sel.xpath('//h1[contains(@class, "mvp-post-title left entry-title")]/text()').get()

print(title)

None


In [16]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
import pandas as pd
import re
import time
import os

# Initialize storage for article data and Parquet file path
article_data = []
parquet_file = "../../data/00-newspaper_data/crawler/tabascohoy/articles.parquet"

# Base URL format for categories
base_url = "https://www.tabascohoy.com/{category}/page/{page}/"

# Define categories to scrape
categories = {
    "Tab": {"cat": "tabasco", "max_pages": 3121},
    "Seguridad": {"cat": "seguridad", "max_pages": 1149},
    "Mexico": {"cat": "mexico", "max_pages": 2770},
    "Mundo": {"cat": "mundo", "max_pages": 871},
    "Dinero": {"cat": "dinero", "max_pages": 152},
    "Opinion": {"cat": "tag/columnista", "max_pages": 580},
    "Deportes": {"cat": "100-deportes", "max_pages": 1255},
    "Like": {"cat": "like", "max_pages": 1129},
    "Aplausos": {"cat": "aplausos", "max_pages": 20},
    "Tecno": {"cat": "tecnologia", "max_pages": 210},
    "Unis": {"cat": "unis-y-mas", "max_pages": 138}
}

# Initialize Selenium WebDriver
options = Options()
options.add_argument("--headless")  # Run in headless mode (no GUI)
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("start-maximized")
options.add_argument("disable-infobars")
options.add_argument("--disable-blink-features=AutomationControlled")

service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)

def is_relevant_url(url):
    """
    Check if the URL matches the pattern YYYY/MM/DD/title.
    """
    pattern = r"https://www\.tabascohoy\.com/[a-zA-Z0-9-]+/?$"
    return re.search(pattern, url)

def extract_article_data(url):
    """
    Extract and return the title, main text, date, and source from an article page.
    """
    try:
        driver.get(url)
        WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "h1")))

        # Extract the title
        try:
            title = driver.find_element(By.CLASS_NAME, "zox-post-title").text.strip()
        except:
            title = None

        # Extract the main text
        try:
            main_text_elements = driver.find_elements(By.CSS_SELECTOR, ".zox-post-main p")
            main_text = ' '.join([p.text.strip() for p in main_text_elements])
        except:
            main_text = None

        # Extract the subtitle
        try:
            subtitle = driver.find_element(By.CLASS_NAME, "zox-post-excerpt").text.strip()
        except:
            subtitle = None

        # Extract the date
        try:
            date = driver.find_element(By.TAG_NAME, "time").get_attribute("datetime")
        except:
            date = None

        # Extract the category
        try:
            category = driver.find_element(By.CLASS_NAME, "zox-post-cat").text.strip()
        except:
            category = None

        return {
            'url': url,
            'title': title,
            'sub_title': subtitle,
            'main_text': main_text,
            'date': date,
            'topic': category
        }

    except Exception as e:
        print(f"Error extracting article from {url}: {e}")
        return None

def crawl_category(category, maxpag):
    """
    Crawl pages 1 to max (manually inputed for each category) in a given category, extracting article links and data.
    """
    for page in range(1, maxpag + 1):
        category_url = base_url.format(category=category, page=page)
        print(f"Accessing category page: {category_url}")

        try:
            driver.get(category_url)
            WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.TAG_NAME, "a")))
            
            # Find and process all article links on the page
            unique_links = set()
            links = driver.find_elements(By.TAG_NAME, "a")
            print(links)
            for link in links:
                print(link)
                href = link.get_attribute("href")
                if href and is_relevant_url(href):
                    unique_links.add(href)

            # Extract article data immediately
            category_data = []
            for link in unique_links:
                article = extract_article_data(link)
                if article:
                    category_data.append(article)
                    print(f"Extracted article from {link}")

            # Save to Parquet
            if category_data:
                save_to_parquet(category_data, parquet_file)

            time.sleep(2)  # Avoid overloading the server

        except Exception as e:
            print(f"Error loading category page {category_url}: {e}")
            continue  # Move to the next page even if this one fails

# Function to save data to Parquet
def save_to_parquet(data, parquet_file):
    df = pd.DataFrame(data)
    if not df.empty:
        if os.path.exists(parquet_file):
            initial = pd.read_parquet(parquet_file)
            df = pd.concat([initial, df]).reset_index(drop=True)
            df.to_parquet(parquet_file, engine="pyarrow", compression="gzip")
        else:
            df.to_parquet(parquet_file, engine="pyarrow", compression="gzip")

# Crawl all categories and extract data
for category, info in categories.items():
    print(f"Starting crawl for category: {category}")
    crawl_category(info['cat'], info['max_pages'])

print("Crawling completed. Data saved in Parquet format.")
driver.quit()


Starting crawl for category: Tab
Accessing category page: https://www.tabascohoy.com/tabasco/page/1/
[<selenium.webdriver.remote.webelement.WebElement (session="7e77089c17cfea879220867d2356ed00", element="f.F5459ECA6C2BE2707F249566D610CDE5.d.8CF1777036A8BC3F2E198C62CA68CAB7.e.5")>]
<selenium.webdriver.remote.webelement.WebElement (session="7e77089c17cfea879220867d2356ed00", element="f.F5459ECA6C2BE2707F249566D610CDE5.d.8CF1777036A8BC3F2E198C62CA68CAB7.e.5")>
Accessing category page: https://www.tabascohoy.com/tabasco/page/2/
[<selenium.webdriver.remote.webelement.WebElement (session="7e77089c17cfea879220867d2356ed00", element="f.F5459ECA6C2BE2707F249566D610CDE5.d.CFAEC20114D91D70FDBCBDA4BFE22AD7.e.12")>]
<selenium.webdriver.remote.webelement.WebElement (session="7e77089c17cfea879220867d2356ed00", element="f.F5459ECA6C2BE2707F249566D610CDE5.d.CFAEC20114D91D70FDBCBDA4BFE22AD7.e.12")>
Accessing category page: https://www.tabascohoy.com/tabasco/page/3/
[<selenium.webdriver.remote.webelemen

KeyboardInterrupt: 

In [19]:
url = 'https://www.tabascohoy.com/ignoro-el-por-que-balearon-a-mi-marido/'
html = requests.get(url).content
html

b'<!DOCTYPE html><html lang="en-US"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131;font-family:system-ui,-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica Neue,Arial,Noto Sans,sans-serif,Apple Color Emoji,Segoe UI Emoji,Segoe UI Symbol,Noto Color Emoji}body{display:flex;flex-direction:column;height:100vh;min-height:100vh}.main-content{margin:8rem auto;max-width:60rem;padding-left:1.5rem}@media (width <= 720px){.main-content{margin-top:4rem}}.h2{font-size:1.5rem;font-weight:500;line-height:2.25rem}@media (width <= 720px){.h2{font-size:1.25rem;line-height:1.5rem}}#challenge-error-text{background-image:url();background-repeat:no-repeat;background

In [7]:

from selenium import webdriver

driver = webdriver.Chrome()
driver.get('https://www.tabascohoy.com/ignoro-el-por-que-balearon-a-mi-marido/')

html = driver.page_source
print(html)  # Now you should see the dynamically loaded content

driver.quit()


<html lang="en-US" dir="ltr"><head><title>Just a moment...</title><meta http-equiv="Content-Type" content="text/html; charset=UTF-8"><meta http-equiv="X-UA-Compatible" content="IE=Edge"><meta name="robots" content="noindex,nofollow"><meta name="viewport" content="width=device-width,initial-scale=1"><style>*{box-sizing:border-box;margin:0;padding:0}html{line-height:1.15;-webkit-text-size-adjust:100%;color:#313131;font-family:system-ui,-apple-system,BlinkMacSystemFont,Segoe UI,Roboto,Helvetica Neue,Arial,Noto Sans,sans-serif,Apple Color Emoji,Segoe UI Emoji,Segoe UI Symbol,Noto Color Emoji}body{display:flex;flex-direction:column;height:100vh;min-height:100vh}.main-content{margin:8rem auto;max-width:60rem;padding-left:1.5rem}@media (width <= 720px){.main-content{margin-top:4rem}}.h2{font-size:1.5rem;font-weight:500;line-height:2.25rem}@media (width <= 720px){.h2{font-size:1.25rem;line-height:1.5rem}}#challenge-error-text{background-image:url();background-repeat:no-repeat;background-size:c

In [1]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pandas as pd
import re
import time
from datetime import datetime, timedelta
import os

# Initialize storage for article data and Parquet file path
article_data = []
parquet_file = "../../data/00-newspaper_data/crawler/qrohoy/articles.parquet"

# Base URL format for categories
base_url = "https://www.tabascohoy.com/{category}/page/{page}/"

# Define categories to scrape
categories = {
    "Tab": {"cat": "tabasco", "max_pages": 3121},
    "Seguridad": {"cat": "seguridad", "max_pages": 1149},
    "Mexico": {"cat": "mexico", "max_pages": 2770},
    "Mundo": {"cat": "mundo", "max_pages": 871},
    "Dinero": {"cat": "dinero", "max_pages": 152},
    "Opinion": {"cat": "tag/columnista", "max_pages": 580},
    "Deportes": {"cat": "100-deportes", "max_pages": 1255},
    "Like": {"cat": "like", "max_pages": 1129},
    "Aplausos": {"cat": "aplausos", "max_pages": 20},
    "Tecno": {"cat": "tecnologia", "max_pages": 210},
    "Unis": {"cat": "unis-y-mas", "max_pages": 138}
}

for category, info in categories.items():
    category_url = base_url.format(category=info['cat'], page=1)  # Construct category URL
    max_pages = info['max_pages']

    print(category_url, max_pages)

https://www.tabascohoy.com/tabasco/page/1/ 3121
https://www.tabascohoy.com/seguridad/page/1/ 1149
https://www.tabascohoy.com/mexico/page/1/ 2770
https://www.tabascohoy.com/mundo/page/1/ 871
https://www.tabascohoy.com/dinero/page/1/ 152
https://www.tabascohoy.com/tag/columnista/page/1/ 580
https://www.tabascohoy.com/100-deportes/page/1/ 1255
https://www.tabascohoy.com/like/page/1/ 1129
https://www.tabascohoy.com/aplausos/page/1/ 20
https://www.tabascohoy.com/tecnologia/page/1/ 210
https://www.tabascohoy.com/unis-y-mas/page/1/ 138
